In [ ]:
#LARGE-SCALE CLIMATE ASSOCIATIONS FOR DDWW


from __future__ import annotations

import os
import glob
import re
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import linregress, pearsonr, spearmanr, rankdata, t as student_t

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
import matplotlib.gridspec as gridspec

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from rasterio.features import geometry_mask
from rasterio.transform import from_origin



# DDWW summary generated by the direct-storage analysis.
DDWW_NC = Path(
    r"E:\R2"
    r"\Landuse_Transition_Analysis_DIRECT_STORAGE_R2"
    r"\transition_landuse_summary_DIRECT_STORAGE_PW_FDR_R2.nc"
)

# TerraClimate direct-storage sources.
SOIL_FILE = Path(
    r"F:\TerraClimate_soil_1958_2023_monthly.nc"
)
SWE_FOLDER = Path(r"F:\Historical data\SWE")
SWE_PATTERN = "TerraClimate_swe_*.nc"

SOIL_VAR = "soil"
SWE_VAR = "swe"

WORLD_SHP = Path(
    r"D:\World_Countries_Generalized.shp"
)

# Climate indices.
PDO_NC = Path(
    r"E:\R2"
    r"\Climate\pdo.timeseries.sstens.nc"
)
AO_TXT = Path(
    r"E:\AO.txt"
)
NOI_CSV = Path(
    r"E:\NOI.csv"
)

# Optional gridded SOI.
SOI_GRID_NC = Path(
    r"E:\R2"
    r"\SOI_gridded_output.nc"
)
USE_SOI_GRID = True

# New output root
BASE_R2_DIR = Path(
    r"E:\R2"
    r"\Climate_Transition_Analysis_DIRECT_STORAGE_R2"
)

OUT_DIR = BASE_R2_DIR / "FINAL_STATISTICS_V8_0p5deg_ICEFREE"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ANNUAL_Z_CACHE = (
    BASE_R2_DIR
    / "annual_direct_storage_1979_2023_0p5deg.nc"
)

P1_START = 1979
P1_END = 2000
P2_START = 2001
P2_END = 2023

TARGET_RESOLUTION_DEG = 0.5
MIN_MONTHS_PER_YEAR = 12
MIN_REFERENCE_YEARS = 18
MIN_REFERENCE_STD = 1.0e-8
SLOPE_EPS = 1.0e-6
FDR_ALPHA = 0.05

# Climate-index settings.
NOI_SEASON_COL = "DJF"
GS_MONTHS = [4, 5, 6, 7, 8, 9]
PHASE_THRESHOLD = 0.5
EXTREME_QUANTILE = 0.75
CORR_FDR_ALPHA = 0.05
MIN_CORR_YEARS = 8
MIN_EFFECTIVE_N = 3.0

SOI_JACCARD_WARN = 0.50

# Figure/output controls.
SAVE_SVG = True
SAVE_PNG = True
SAVE_PDF = True
FIG_DPI = 500
MAP_RASTERIZED_IN_VECTOR = True
PUB_FIGSIZE = (16.5, 11.7)

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 8,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})


STATE_CODE = {"WW": 1, "DD": 2, "WD": 3, "DW": 4}
STATE_ORDER = ["WW", "DD", "WD", "DW"]

STATE_COLORS = {
    "WW": "#1a9850",
    "DD": "#b2182b",
    "WD": "#74add1",
    "DW": "#fdae61",
}

CLASS_COLORS_TRANS = {1:"#1a9850", 2:"#b2182b", 3:"#74add1", 4:"#fdae61"}

CLASS_COLORS_BASE = {
    1: "#9ecae1",  # wetting
    2: "#f4a6a6",  # drying
}


def save_figure(fig, out_base, dpi=500):
    out_base = str(out_base)
    if SAVE_SVG:
        fig.savefig(
            out_base + ".svg",
            format="svg",
            dpi=dpi,
            bbox_inches="tight",
            facecolor="white",
        )
    if SAVE_PNG:
        fig.savefig(
            out_base + ".png",
            format="png",
            dpi=dpi,
            bbox_inches="tight",
            facecolor="white",
        )
    if SAVE_PDF:
        fig.savefig(
            out_base + ".pdf",
            format="pdf",
            dpi=dpi,
            bbox_inches="tight",
            facecolor="white",
        )


def safe_to_csv(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print("Saved:", path)


def normalize_lat_lon(da):
    rename = {}
    if "latitude" in da.dims:
        rename["latitude"] = "lat"
    if "longitude" in da.dims:
        rename["longitude"] = "lon"
    if rename:
        da = da.rename(rename)

    if "lat" not in da.dims or "lon" not in da.dims:
        raise ValueError(f"Expected lat/lon dimensions; found {da.dims}")

    if float(da["lon"].max()) > 180.0:
        da = da.assign_coords(
            lon=((da["lon"] + 180.0) % 360.0) - 180.0
        ).sortby("lon")

    return da.sortby("lat").sortby("lon")


def make_target_centers(res=1.0):
    lat = np.arange(-90.0 + res / 2.0, 90.0, res, dtype=np.float64)
    lon = np.arange(-180.0 + res / 2.0, 180.0, res, dtype=np.float64)
    return lat, lon


TARGET_LAT, TARGET_LON = make_target_centers(TARGET_RESOLUTION_DEG)


def infer_coarsen_factor(coord, target_res):
    values = np.asarray(coord.values, dtype=np.float64)
    d = float(np.nanmedian(np.abs(np.diff(values))))
    if not np.isfinite(d) or d <= 0:
        raise ValueError("Could not infer native spatial resolution.")

    factor = int(round(target_res / d))
    if factor < 1:
        factor = 1

    if abs(factor * d - target_res) > max(1e-4, 0.03 * target_res):
        raise ValueError(
            f"Native spacing {d:.8f} does not coarsen cleanly to "
            f"{target_res} degree. factor={factor}"
        )
    return factor


def annual_mean_monthly_state(storage):
    storage = storage.sel(
        time=slice(f"{P1_START}-01-01", f"{P2_END}-12-31")
    )

    years = np.asarray(storage["time"].dt.year.values)
    counts = pd.Series(years).value_counts().sort_index()
    bad = counts[counts < MIN_MONTHS_PER_YEAR]
    if len(bad):
        raise ValueError(
            "Insufficient monthly records: "
            + ", ".join(f"{int(y)} ({int(n)})" for y, n in bad.items())
        )

    annual = storage.groupby("time.year").mean("time", skipna=True)
    annual = annual.sel(year=slice(P1_START, P2_END))
    annual.name = "annual_direct_storage"
    return annual


def regrid_annual_to_target(annual_native):
    lat_factor = infer_coarsen_factor(
        annual_native["lat"], TARGET_RESOLUTION_DEG
    )
    lon_factor = infer_coarsen_factor(
        annual_native["lon"], TARGET_RESOLUTION_DEG
    )

    print(
        f"Native-to-target coarsening factors: "
        f"lat={lat_factor}, lon={lon_factor}"
    )

    # Compatibility with older xarray: do not pass skipna= here.
    coarse = annual_native.coarsen(
        lat=lat_factor,
        lon=lon_factor,
        boundary="trim",
    ).mean()

    return coarse.interp(
        lat=TARGET_LAT,
        lon=TARGET_LON,
        method="nearest",
    ).astype("float32")


def build_fixed_reference(early_annual):
    n = early_annual.count("year")
    mean = early_annual.mean("year", skipna=True)
    std = early_annual.std("year", skipna=True, ddof=1)

    good = (
        (n >= MIN_REFERENCE_YEARS)
        & np.isfinite(std)
        & (std >= MIN_REFERENCE_STD)
    )

    return (
        mean.where(good).astype("float32"),
        std.where(good).astype("float32"),
        n.where(good, 0).astype("int16"),
    )


def standardize_annual(annual, ref_mean, ref_std):
    z = ((annual - ref_mean) / ref_std).astype("float32")
    z.name = "annual_standardized_storage"
    z.attrs["definition"] = (
        "(annual soil+SWE - 1979-2000 mean) / 1979-2000 sample sd"
    )
    return z


def area_weighted_class_percent(class_da, classes):
    weights_2d = AREA_WEIGHTS
    _strict_xy(class_da,weights_2d,"weighted class statistics")

    valid = np.isfinite(class_da)
    denom = weights_2d.where(valid).sum(("lat", "lon"), skipna=True)

    rows = []
    for code, label in classes:
        num = weights_2d.where(class_da == code).sum(
            ("lat", "lon"), skipna=True
        )
        pct = (
            float((100.0 * num / denom).values)
            if float(denom.values) > 0 else np.nan
        )
        rows.append({
            "class_code": code,
            "class_label": label,
            "area_weighted_percent": pct,
        })

    return pd.DataFrame(rows)


def setup_map_ax(ax):
    ax.set_global()
    ax.coastlines(linewidth=0.45)
    ax.add_feature(
        cfeature.BORDERS,
        linestyle="-",
        linewidth=0.25,
        alpha=0.45,
    )
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.25,
        color="gray",
        alpha=0.35,
        linestyle="--",
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 7}
    gl.ylabel_style = {"size": 7}


def plot_mask(
    ax,
    lon_vals,
    lat_vals,
    mask,
    marker="+",
    size=2.0,
    color="black",
    linewidths=0.25,
    alpha=0.85,
    step=3,
    zorder=5,
):
    arr = np.asarray(mask.values, dtype=bool)
    iy, ix = np.where(arr)

    if len(ix) == 0:
        return

    iy = iy[::step]
    ix = ix[::step]

    ax.scatter(
        lon_vals[ix],
        lat_vals[iy],
        marker=marker,
        s=size,
        color=color,
        linewidths=linewidths,
        alpha=alpha,
        transform=ccrs.PlateCarree(),
        zorder=zorder,
    )


def add_panel_label(ax, label, x=0.01, y=0.98):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.75,
            pad=2.5,
        ),
        zorder=20,
    )


def detrend_series(year, values):
    year = np.asarray(year, dtype=float)
    values = np.asarray(values, dtype=float)

    out = np.full(values.shape, np.nan, dtype=float)
    ok = np.isfinite(year) & np.isfinite(values)

    if ok.sum() < 3:
        return out

    res = linregress(year[ok], values[ok])
    out[ok] = values[ok] - (res.intercept + res.slope * year[ok])
    return out


def zscore_index_dataframe(df, value_col):
    """
    Standardize a climate-index time series over the available study years.
    """
    out = df[["year", value_col]].copy()
    vals = pd.to_numeric(out[value_col], errors="coerce").astype(float)

    mu = float(vals.mean(skipna=True))
    sd = float(vals.std(skipna=True, ddof=1))

    if not np.isfinite(sd) or sd <= 0:
        raise ValueError(
            f"Cannot standardize climate index {value_col}: mean={mu}, sd={sd}"
        )

    out[f"{value_col}_z"] = (vals - mu) / sd
    return out, mu, sd


def lag1_autocorr(values):
    """
    Lag-1 Pearson autocorrelation using consecutive finite observations.
    """
    x = np.asarray(values, dtype=float)

    if x.size < 4:
        return np.nan

    x0 = x[:-1]
    x1 = x[1:]
    ok = np.isfinite(x0) & np.isfinite(x1)

    if ok.sum() < 3:
        return np.nan

    if np.nanstd(x0[ok]) == 0 or np.nanstd(x1[ok]) == 0:
        return 0.0

    return float(np.corrcoef(x0[ok], x1[ok])[0, 1])


def effective_sample_size_ar1(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]
    n = int(len(x))

    if n < 4:
        return np.nan, np.nan, np.nan

    rho_x = lag1_autocorr(x)
    rho_y = lag1_autocorr(y)

    if not np.isfinite(rho_x):
        rho_x = 0.0
    if not np.isfinite(rho_y):
        rho_y = 0.0

    prod = float(np.clip(rho_x * rho_y, -0.95, 0.95))
    neff = n * (1.0 - prod) / (1.0 + prod)
    neff = float(np.clip(neff, MIN_EFFECTIVE_N, float(n)))

    return neff, rho_x, rho_y


def correlation_p_from_neff(r, neff):
    """
    Two-sided correlation p-value using AR(1)-adjusted effective sample size.
    """
    if not np.isfinite(r) or not np.isfinite(neff) or neff <= 2:
        return np.nan

    r = float(np.clip(r, -0.999999999, 0.999999999))
    df = float(neff - 2.0)

    t_stat = r * np.sqrt(df / max(1.0e-15, 1.0 - r * r))
    return float(2.0 * student_t.sf(abs(t_stat), df=df))


def benjamini_hochberg_qvalues(values):
    """
    Benjamini-Hochberg FDR q-values.
    """
    p = np.asarray(values, dtype=float)
    q = np.full(p.shape, np.nan, dtype=float)

    valid = np.isfinite(p)
    pv = p[valid]

    if pv.size == 0:
        return q

    order = np.argsort(pv)
    ranked = pv[order]
    m = ranked.size

    q_ranked = ranked * m / np.arange(1, m + 1, dtype=float)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0.0, 1.0)

    q_valid = np.empty_like(q_ranked)
    q_valid[order] = q_ranked
    q[valid] = q_valid

    return q


def apply_fdr_to_corr_table(df):
   
    out = df.copy()

    out["pearson_q_ar1_fdr"] = benjamini_hochberg_qvalues(
        out["pearson_p_ar1"].values
    )

    out["spearman_q_ar1_fdr"] = benjamini_hochberg_qvalues(
        out["spearman_p_ar1"].values
    )

    return out


def pairwise_year_overlap_table(extreme_years):
    """
    Pairwise selected-year overlap diagnostics for SOI composites.
    """
    rows = []
    states = list(STATE_ORDER)

    for i, a in enumerate(states):
        set_a = set(extreme_years.get(a, []))

        for b in states[i:]:
            set_b = set(extreme_years.get(b, []))
            inter = set_a & set_b
            union = set_a | set_b

            jaccard = (
                len(inter) / len(union)
                if len(union) > 0 else np.nan
            )

            rows.append({
                "state_1": a,
                "state_2": b,
                "n_state_1": len(set_a),
                "n_state_2": len(set_b),
                "n_intersection": len(inter),
                "jaccard": jaccard,
                "shared_years": ",".join(map(str, sorted(inter))),
            })

    return pd.DataFrame(rows)


def soi_composite_spatial_correlation(composites):
    """
    Pairwise spatial Pearson correlation among SOI composite maps.
    """
    rows = []
    states = list(STATE_ORDER)

    for i, a in enumerate(states):
        ca = composites.get(a)

        for b in states[i:]:
            cb = composites.get(b)

            if ca is None or cb is None:
                r = np.nan
                n = 0
            else:
                va = np.asarray(ca.values, dtype=float).ravel()
                vb = np.asarray(cb.values, dtype=float).ravel()
                ok = np.isfinite(va) & np.isfinite(vb)
                n = int(ok.sum())

                if n >= 3 and np.nanstd(va[ok]) > 0 and np.nanstd(vb[ok]) > 0:
                    r = float(np.corrcoef(va[ok], vb[ok])[0, 1])
                else:
                    r = np.nan

            rows.append({
                "state_1": a,
                "state_2": b,
                "n_gridcells": n,
                "spatial_r": r,
            })

    return pd.DataFrame(rows)

FINAL_ROOT = Path(r"F:\Derived")
FINAL_V8 = FINAL_ROOT / "DDWW_SEGMENTED_HAC_ICEFREE" / "historical_45yr_segmented_HAC_results.nc"
FINAL_AREA = FINAL_ROOT / "DIRECT_STORAGE_STATE_TRENDS_MC_CALIBRATED_v3" / "CORRECTED_AREA" / "TerraClimate_actual_land_area_0p5deg.nc"
FINAL_ICE = FINAL_ROOT / "DDWW_INDEPENDENT_ICE_MASK_v5" / "NaturalEarth_10m_glaciated_0p5deg_exclude_ge_5percent.nc"
FINAL_STATE = FINAL_ROOT / "SPEI_SR" / "STORAGE_CLOSURE" / "TerraClimate_soil_plus_swe_0p5deg_1958_2023.nc"
EXCLUDE_SOUTH_OF_60 = True


def _to_lon180(ds):
    renamed={}
    for a,b in (("latitude","lat"),("longitude","lon")):
        if a in ds.dims: renamed[a]=b
    if renamed:ds=ds.rename(renamed)
    if not all(z in ds.coords for z in ("lat","lon")):
        raise ValueError("Input must have lat and lon dimensions")
    longitude=((ds.lon.astype(float)+180.0)%360.0)-180.0
    ds=ds.assign_coords(lon=longitude).sortby("lon").sortby("lat")
    if len(np.unique(np.round(ds.lon.values,7))) != ds.sizes['lon']:
        raise ValueError("Duplicate longitudes after conversion")
    return ds


def _strict_xy(reference,other,description):
    for dim in ("lat","lon"):
        x=np.asarray(reference[dim].values,dtype=float)
        y=np.asarray(other[dim].values,dtype=float)
        if len(x)!=len(y) or not np.allclose(x,y,atol=1e-7,rtol=0):
            raise RuntimeError(f"{description}: incompatible {dim} coordinates, no silent resampling")


def load_final_v8_landuse_fields():
  
    for f in (FINAL_V8,FINAL_AREA,FINAL_ICE):
        if not f.is_file():raise FileNotFoundError(f"Required FINAL input missing: {f}")
    with xr.open_dataset(FINAL_V8) as ds0, xr.open_dataset(FINAL_AREA) as ar0, xr.open_dataset(FINAL_ICE) as ice0:
        ds,ar,ice=map(_to_lon180,(ds0,ar0,ice0))
        _strict_xy(ds,ar,"v3 actual land area")
        _strict_xy(ds,ice,"independent glacier mask")
        for z in ("descriptive_class","classifiable","valid_icefree_land","slope_p1","slope_p2","p_calibrated_L3"):
            if z not in ds:raise KeyError(f"v8 requires {z}; available: {list(ds.data_vars)}")
        area=ar["actual_land_area"].transpose("lat","lon").load().astype("float64")
        glacier=ice["permanent_ice"].transpose("lat","lon").load()
        if not np.isin(glacier.values,[0,1]).all():raise ValueError("Permanent ice must be binary")
        original_domain=(area>0)&np.isfinite(area)&(glacier==0)
        v8_domain=ds['valid_icefree_land'].transpose('lat','lon').load().astype(bool)
        if not np.array_equal(original_domain.values,v8_domain.values):
            raise RuntimeError("V8 stored domain and revised v3 area/independent glacier mask differ. Reconcile first.")
        domain=original_domain & (area.lat>=-60 if EXCLUDE_SOUTH_OF_60 else True)
        codes=ds['descriptive_class'].transpose('lat','lon').load()
        classifiable=ds['classifiable'].transpose('lat','lon').load().astype(bool)
        valid=domain&classifiable&codes.isin([1,2,3,4])
        s1=ds['slope_p1'].transpose('lat','lon').load().astype(float)
        s2=ds['slope_p2'].transpose('lat','lon').load().astype(float)
        # Verify correct v8 sign mapping, independent of old 1-degree output.
        expected=xr.full_like(codes,0)
        expected=xr.where((s1<0)&(s2<0),1,expected)
        expected=xr.where((s1>0)&(s2>0),2,expected)
        expected=xr.where((s1>0)&(s2<0),3,expected)
        expected=xr.where((s1<0)&(s2>0),4,expected)
        if bool(((codes!=expected)&valid).any().item()):raise RuntimeError("Stored v8 class codes do not equal slope signs")
        legacy=xr.where(codes==1,2,xr.where(codes==2,1,codes))
        legacy=legacy.where(valid).astype('float32').rename('DDWW_class_legacy_figure_order')
        pchange=ds['p_calibrated_L3'].transpose('lat','lon').load()
        p1=s1.where(valid)
        area=area.where(domain).rename('actual_land_area_m2')
        domain_area=float(area.sum(skipna=True).item())
        classified_area=float(area.where(valid).sum(skipna=True).item())
        if classified_area <= 0:raise ValueError('No v8 classifiable ice-free area')
        print(f'Final V8 0.5 degree domain: classified {100*classified_area/domain_area:.2f}% of eligible ice-free land')
        print('Strictly descriptive sign classes; v8 p_calibrated_L3 tests SLOPE CHANGE only.')
        return legacy,area,p1,pchange.where(valid),valid,domain


print('Historical 0.5-degree v8 DDWW classes and actual land area ...')
class_trans, AREA_WEIGHTS, p1_slope, p_change_v8, valid_transition, land_mask = load_final_v8_landuse_fields()
class_base = xr.full_like(p1_slope,np.nan,dtype=np.float32)
class_base=xr.where((p1_slope>0)&valid_transition,1.,class_base)
class_base=xr.where((p1_slope<0)&valid_transition,2.,class_base)
# No period-1 significance claim: v8 tests ONLY the breakpoint slope change.
sig_mask_p1_fdr = xr.zeros_like(class_base,dtype=bool)
sig_mask_transition_fdr = (p_change_v8 < 0.05) & valid_transition
pcts_base_df=area_weighted_class_percent(class_base,[(1,'Wetting'),(2,'Drying')])
pcts_trans_df=area_weighted_class_percent(class_trans,[(1,'WW'),(2,'DD'),(3,'WD'),(4,'DW')])
safe_to_csv(pcts_base_df,OUT_DIR/'area_weighted_P1_direction_percentages.csv')
safe_to_csv(pcts_trans_df,OUT_DIR/'area_weighted_DDWW_sign_class_percentages.csv')
print(pcts_trans_df.round(3))


def build_annual_z_cache():
    """Use same 0.5-degree stored state used for v8, not legacy 1-degree resampling."""
    if not FINAL_STATE.is_file():raise FileNotFoundError(FINAL_STATE)
    with xr.open_dataset(FINAL_STATE, chunks={'time':12}) as sd:
        if 'soil_swe_storage' not in sd: raise KeyError('soil_swe_storage missing')
        ds=_to_lon180(sd)
        monthly=ds['soil_swe_storage'].sel(time=slice('1979-01','2023-12'))
        if monthly.sizes.get('time',0)!=540:raise ValueError('Need 540 monthly records from 1979 to 2023')
        yy=np.asarray(monthly.time.dt.year.values,dtype=int)
        mm=np.asarray(monthly.time.dt.month.values,dtype=int)
        if not np.array_equal(yy,np.repeat(np.arange(1979,2024),12)) or not np.array_equal(mm,np.tile(np.arange(1,13),45)):
            raise ValueError('Monthly time axis gaps/duplicates in 1979-2023')
        _strict_xy(class_trans,monthly,'monthly 0.5-degree modeled storage')
        yearly_count=monthly.notnull().groupby('time.year').sum('time')
        yearly_sum=monthly.fillna(0).groupby('time.year').sum('time')
        annual_storage=(yearly_sum/12.).where(yearly_count==12).where(land_mask).astype('float32').load()
    ref_mean,ref_std,ref_n=build_fixed_reference(annual_storage.sel(year=slice(1979,2000)))
    annual_z=standardize_annual(annual_storage,ref_mean,ref_std)
    cache_ds=xr.Dataset({'annual_storage_mm':annual_storage,
                          'annual_z':annual_z,
                          'reference_mean_1979_2000':ref_mean,
                          'reference_std_1979_2000':ref_std,
                          'reference_n_1979_2000':ref_n})
    cache_ds.attrs.update({'storage_definition':'TerraClimate modeled soil + SWE',
                          'grid':'0.5-degree exact v8 coordinates',
                          'annual_completeness':'exactly 12 monthly storage values per pixel'})
    cache_ds.to_netcdf(ANNUAL_Z_CACHE)
    print('Saved v8-compatible annual 0.5-degree cache:',ANNUAL_Z_CACHE)
    return annual_storage, annual_z

if ANNUAL_Z_CACHE.exists():
    print("Loading cached annual direct storage...")
    ds_cache = xr.open_dataset(ANNUAL_Z_CACHE)
    annual_z = ds_cache["annual_z"]

    if "annual_storage_mm" in ds_cache:
        annual_storage = ds_cache["annual_storage_mm"]
    else:
        print(
            "Existing cache has no annual_storage_mm; reconstructing raw "
            "annual storage from cached z, reference mean, and reference sd."
        )
        annual_storage = (
            ds_cache["annual_z"]
            * ds_cache["reference_std_1979_2000"]
            + ds_cache["reference_mean_1979_2000"]
        ).astype("float32")
else:
    annual_storage, annual_z = build_annual_z_cache()

annual_storage = annual_storage.sel(year=slice(P1_START, P2_END))
_strict_xy(class_trans, annual_storage, "annual-storage cache")
annual_storage=annual_storage.where(land_mask)

annual_z = annual_z.sel(year=slice(P1_START, P2_END))
_strict_xy(class_trans, annual_z, "annual-z cache")
annual_z=annual_z.where(land_mask)



print(
    "Computing area-weighted CLASS-MEAN annual direct storage first, "
    "then standardizing each class-level time series..."
)

weights_2d = AREA_WEIGHTS
_strict_xy(annual_storage,weights_2d,"annual series weights")

years = np.asarray(annual_storage["year"].values, dtype=int)
class_ts = pd.DataFrame({"year": years})
class_reference_rows = []

for st in STATE_ORDER:
    code = STATE_CODE[st]
    class_mask = (class_trans == code) & land_mask

    # Static class area.
    w_static = weights_2d.where(class_mask)
    total_class_weight = float(
        w_static.sum(("lat", "lon"), skipna=True).values
    )

    # Year-specific valid-area denominator.
    valid_storage = np.isfinite(annual_storage)
    w_valid = w_static.where(valid_storage)

    denom_year = w_valid.sum(
        ("lat", "lon"),
        skipna=True,
    )

    numerator_year = (
        annual_storage * w_static
    ).sum(
        ("lat", "lon"),
        skipna=True,
    )

    class_mean_mm = numerator_year / denom_year

    class_ts[f"{st}_mean_mm"] = np.asarray(
        class_mean_mm.values,
        dtype=float,
    )

    if total_class_weight > 0:
        valid_fraction = denom_year / total_class_weight
        class_ts[f"{st}_valid_area_fraction"] = np.asarray(
            valid_fraction.values,
            dtype=float,
        )
    else:
        class_ts[f"{st}_valid_area_fraction"] = np.nan

   
    p1_values = class_ts.loc[
        class_ts["year"].between(P1_START, P1_END),
        f"{st}_mean_mm",
    ].astype(float)

    ref_mean_class = float(p1_values.mean(skipna=True))
    ref_std_class = float(p1_values.std(skipna=True, ddof=1))
    ref_n_class = int(p1_values.notna().sum())

    if (
        ref_n_class < MIN_REFERENCE_YEARS
        or not np.isfinite(ref_std_class)
        or ref_std_class <= 0
    ):
        raise ValueError(
            f"Invalid class-level reference for {st}: "
            f"n={ref_n_class}, mean={ref_mean_class}, sd={ref_std_class}"
        )

    class_ts[f"{st}_z"] = (
        class_ts[f"{st}_mean_mm"] - ref_mean_class
    ) / ref_std_class

    class_ts[f"{st}_detrended"] = detrend_series(
        class_ts["year"].values,
        class_ts[f"{st}_z"].values,
    )

    class_reference_rows.append({
        "state": st,
        "reference_period": f"{P1_START}-{P1_END}",
        "reference_mean_mm": ref_mean_class,
        "reference_sd_mm": ref_std_class,
        "reference_n_years": ref_n_class,
        "minimum_annual_valid_area_fraction": float(
            np.nanmin(class_ts[f"{st}_valid_area_fraction"].values)
        ),
        "maximum_absolute_class_z": float(
            np.nanmax(np.abs(class_ts[f"{st}_z"].values))
        ),
    })

class_reference_df = pd.DataFrame(class_reference_rows)

safe_to_csv(
    class_ts,
    OUT_DIR / "annual_class_mean_direct_storage_corrected.csv",
)

safe_to_csv(
    class_reference_df,
    OUT_DIR / "annual_class_storage_reference_diagnostics.csv",
)

print("\nClass-level storage diagnostics:")
print(class_reference_df.round(4).to_string(index=False))

for _, row in class_reference_df.iterrows():
    if row["minimum_annual_valid_area_fraction"] < 0.95:
        print(
            f"WARNING: {row['state']} minimum valid area coverage is "
            f"{100*row['minimum_annual_valid_area_fraction']:.1f}%."
        )

    if row["maximum_absolute_class_z"] > 6.0:
        print(
            f"WARNING: {row['state']} still exceeds |z|=6 after "
            "class-level standardization. Inspect source data before "
            "interpreting the climate associations."
        )


def read_ao_txt(path):
    rows = []

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                try:
                    y = int(parts[0])
                    m = int(parts[1])
                    v = float(parts[2])
                    rows.append([y, m, v])
                except Exception:
                    pass

    df = pd.DataFrame(
        rows,
        columns=["year", "month", "ao"],
    )

    df["date"] = pd.to_datetime(
        dict(
            year=df["year"],
            month=df["month"],
            day=1,
        )
    )

    return df.sort_values("date")


def read_noi_csv(path, season_col="DJF"):
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]

    year_col = None
    for c in ["Year", "year", "YEAR"]:
        if c in df.columns:
            year_col = c
            break

    if year_col is None:
        raise ValueError(
            f"No year column found in NOI CSV. Columns={list(df.columns)}"
        )

    df = df.rename(columns={year_col: "year"})
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    if season_col not in df.columns:
        raise ValueError(
            f"{season_col} not found in NOI columns: {list(df.columns)}"
        )

    df[season_col] = pd.to_numeric(
        df[season_col],
        errors="coerce",
    )

    out = df[["year", season_col]].dropna().copy()
    out = out.rename(columns={season_col: "noi"})
    out["year"] = out["year"].astype(int)

    return out.sort_values("year")


def annual_mean_monthly_index(df, value_col):
    x = df.copy()
    x["year"] = x["date"].dt.year
    return x.groupby("year", as_index=False)[value_col].mean()


def gs_mean_monthly_index(df, value_col, months):
    x = df[df["date"].dt.month.isin(months)].copy()
    x["year"] = x["date"].dt.year
    return x.groupby("year", as_index=False)[value_col].mean()


print("Loading PDO, AO and NOI...")

if not PDO_NC.exists():
    raise FileNotFoundError(PDO_NC)
if not AO_TXT.exists():
    raise FileNotFoundError(AO_TXT)
if not NOI_CSV.exists():
    raise FileNotFoundError(NOI_CSV)

pdo_ds = xr.open_dataset(PDO_NC)

if "pdo" in pdo_ds.data_vars:
    pdo_var = "pdo"
else:
    pdo_var = list(pdo_ds.data_vars)[0]

pdo_df = pd.DataFrame({
    "date": pd.to_datetime(pdo_ds["time"].values),
    "pdo": np.asarray(pdo_ds[pdo_var].values, dtype=float),
}).dropna()

ao_df = read_ao_txt(AO_TXT)
noi_df = read_noi_csv(NOI_CSV, NOI_SEASON_COL)

pdo_ann = annual_mean_monthly_index(pdo_df, "pdo")
ao_ann = annual_mean_monthly_index(ao_df, "ao")
noi_ann = noi_df.copy()

pdo_gs = gs_mean_monthly_index(pdo_df, "pdo", GS_MONTHS)
ao_gs = gs_mean_monthly_index(ao_df, "ao", GS_MONTHS)
noi_gs = noi_ann.copy()

for df in [pdo_ann, ao_ann, noi_ann, pdo_gs, ao_gs, noi_gs]:
    df.drop(
        df[
            (df["year"] < P1_START)
            | (df["year"] > P2_END)
        ].index,
        inplace=True,
    )

climate_indices_annual = (
    pdo_ann
    .merge(ao_ann, on="year", how="outer")
    .merge(noi_ann, on="year", how="outer")
    .sort_values("year")
)

climate_indices_gs = (
    pdo_gs
    .merge(ao_gs, on="year", how="outer")
    .merge(noi_gs, on="year", how="outer")
    .sort_values("year")
)


pdo_ann_z, pdo_ann_mean, pdo_ann_sd = zscore_index_dataframe(pdo_ann, "pdo")
ao_ann_z, ao_ann_mean, ao_ann_sd = zscore_index_dataframe(ao_ann, "ao")
noi_ann_z, noi_ann_mean, noi_ann_sd = zscore_index_dataframe(noi_ann, "noi")

pdo_gs_z, pdo_gs_mean, pdo_gs_sd = zscore_index_dataframe(pdo_gs, "pdo")
ao_gs_z, ao_gs_mean, ao_gs_sd = zscore_index_dataframe(ao_gs, "ao")
noi_gs_z, noi_gs_mean, noi_gs_sd = zscore_index_dataframe(noi_gs, "noi")

climate_index_scaling = pd.DataFrame([
    ["annual", "PDO", pdo_ann_mean, pdo_ann_sd],
    ["annual", "AO", ao_ann_mean, ao_ann_sd],
    ["annual", "NOI", noi_ann_mean, noi_ann_sd],
    ["growing_season", "PDO", pdo_gs_mean, pdo_gs_sd],
    ["growing_season", "AO", ao_gs_mean, ao_gs_sd],
    ["growing_season", "NOI", noi_gs_mean, noi_gs_sd],
], columns=[
    "analysis",
    "index",
    "study_period_mean",
    "study_period_sd",
])

safe_to_csv(
    climate_index_scaling,
    OUT_DIR / "climate_index_standardization_parameters.csv",
)

climate_indices_annual_z = (
    pdo_ann_z[["year", "pdo_z"]]
    .merge(ao_ann_z[["year", "ao_z"]], on="year", how="outer")
    .merge(noi_ann_z[["year", "noi_z"]], on="year", how="outer")
    .sort_values("year")
)

climate_indices_gs_z = (
    pdo_gs_z[["year", "pdo_z"]]
    .merge(ao_gs_z[["year", "ao_z"]], on="year", how="outer")
    .merge(noi_gs_z[["year", "noi_z"]], on="year", how="outer")
    .sort_values("year")
)

safe_to_csv(
    climate_indices_annual_z,
    OUT_DIR / "climate_indices_annual_standardized.csv",
)

safe_to_csv(
    climate_indices_gs_z,
    OUT_DIR / "climate_indices_growing_season_standardized.csv",
)

safe_to_csv(
    climate_indices_annual,
    OUT_DIR / "climate_indices_annual.csv",
)

safe_to_csv(
    climate_indices_gs,
    OUT_DIR / "climate_indices_growing_season.csv",
)


def corr_table(storage_df, idx_df, value_col, index_label):
    rows = []
    merged = storage_df.merge(idx_df, on="year", how="inner")

    for st in STATE_ORDER:
        x = np.asarray(
            merged[f"{st}_detrended"].values,
            dtype=float,
        )
        y = np.asarray(
            merged[value_col].values,
            dtype=float,
        )

        ok = np.isfinite(x) & np.isfinite(y)
        x_ok = x[ok]
        y_ok = y[ok]
        n = int(ok.sum())

        if n >= MIN_CORR_YEARS:
            pearson_r = float(pearsonr(x_ok, y_ok).statistic)
            spearman_r = float(spearmanr(x_ok, y_ok).statistic)

            neff_p, rho_x_p, rho_y_p = effective_sample_size_ar1(
                x_ok,
                y_ok,
            )
            pearson_p_ar1 = correlation_p_from_neff(
                pearson_r,
                neff_p,
            )

            # For Spearman inference, estimate AR(1) after ranking both series.
            x_rank = rankdata(x_ok)
            y_rank = rankdata(y_ok)

            neff_s, rho_x_s, rho_y_s = effective_sample_size_ar1(
                x_rank,
                y_rank,
            )
            spearman_p_ar1 = correlation_p_from_neff(
                spearman_r,
                neff_s,
            )
        else:
            pearson_r = np.nan
            spearman_r = np.nan
            pearson_p_ar1 = np.nan
            spearman_p_ar1 = np.nan
            neff_p = np.nan
            neff_s = np.nan
            rho_x_p = np.nan
            rho_y_p = np.nan
            rho_x_s = np.nan
            rho_y_s = np.nan

        rows.append({
            "state": st,
            "index": index_label,
            "value_column": value_col,
            "n": n,
            "pearson_r": pearson_r,
            "pearson_p_ar1": pearson_p_ar1,
            "pearson_neff": neff_p,
            "storage_ar1_for_pearson": rho_x_p,
            "index_ar1_for_pearson": rho_y_p,
            "spearman_r": spearman_r,
            "spearman_p_ar1": spearman_p_ar1,
            "spearman_neff": neff_s,
            "storage_rank_ar1": rho_x_s,
            "index_rank_ar1": rho_y_s,
        })

    return pd.DataFrame(rows)


def build_phase_composite_table(
    storage_df,
    idx_df,
    value_col,
    index_label,
    thr=0.5,
):
    merged = storage_df.merge(idx_df, on="year", how="inner")

    pos = merged[merged[value_col] >= thr]
    neg = merged[merged[value_col] <= -thr]

    rows = []

    for st in STATE_ORDER:
        col = f"{st}_detrended"

        mean_pos = (
            float(pos[col].mean())
            if len(pos) > 0 else np.nan
        )

        mean_neg = (
            float(neg[col].mean())
            if len(neg) > 0 else np.nan
        )

        rows.append({
            "state": st,
            "index": index_label,
            "standardized_index_column": value_col,
            "phase_threshold_z": thr,
            "mean_detrended_storage_pos_phase": mean_pos,
            "mean_detrended_storage_neg_phase": mean_neg,
            "pos_minus_neg": (
                mean_pos - mean_neg
                if np.isfinite(mean_pos)
                and np.isfinite(mean_neg)
                else np.nan
            ),
            "n_pos": int(len(pos)),
            "n_neg": int(len(neg)),
            "positive_years": ",".join(
                map(str, pos["year"].astype(int).tolist())
            ),
            "negative_years": ",".join(
                map(str, neg["year"].astype(int).tolist())
            ),
        })

    return pd.DataFrame(rows)


corr_annual = pd.concat([
    corr_table(class_ts, pdo_ann_z, "pdo_z", "pdo"),
    corr_table(class_ts, ao_ann_z, "ao_z", "ao"),
    corr_table(class_ts, noi_ann_z, "noi_z", "noi"),
], ignore_index=True)

corr_gs = pd.concat([
    corr_table(class_ts, pdo_gs_z, "pdo_z", "pdo"),
    corr_table(class_ts, ao_gs_z, "ao_z", "ao"),
    corr_table(class_ts, noi_gs_z, "noi_z", "noi"),
], ignore_index=True)

corr_annual = apply_fdr_to_corr_table(corr_annual)
corr_gs = apply_fdr_to_corr_table(corr_gs)

phase_annual = pd.concat([
    build_phase_composite_table(
        class_ts,
        pdo_ann_z,
        "pdo_z",
        "pdo",
        PHASE_THRESHOLD,
    ),
    build_phase_composite_table(
        class_ts,
        ao_ann_z,
        "ao_z",
        "ao",
        PHASE_THRESHOLD,
    ),
    build_phase_composite_table(
        class_ts,
        noi_ann_z,
        "noi_z",
        "noi",
        PHASE_THRESHOLD,
    ),
], ignore_index=True)

safe_to_csv(
    corr_annual,
    OUT_DIR
    / "correlations_annual_AR1_effectiveN_BH_FDR.csv",
)

safe_to_csv(
    corr_gs,
    OUT_DIR
    / "correlations_growing_season_AR1_effectiveN_BH_FDR.csv",
)

safe_to_csv(
    phase_annual,
    OUT_DIR
    / "phase_composites_standardized_indices_z0p5.csv",
)

print("\nAnnual climate correlations with AR(1)-adjusted p and BH-FDR q:")
print(
    corr_annual[
        [
            "state",
            "index",
            "n",
            "pearson_r",
            "pearson_neff",
            "pearson_p_ar1",
            "pearson_q_ar1_fdr",
        ]
    ].round(4).to_string(index=False)
)


def normalize_lon(da):
    return normalize_lat_lon(da)


soi_comps = None
soi_years_df = None
soi_vmin = None
soi_vmax = None

if USE_SOI_GRID and SOI_GRID_NC.exists():
    print("Preparing revised SOI composites...")

    soi_ds = xr.open_dataset(SOI_GRID_NC)

    if "SOI" in soi_ds.data_vars:
        soi_var = "SOI"
    else:
        soi_var = list(soi_ds.data_vars)[0]

    soi = normalize_lon(soi_ds[soi_var])

    try:
        soi = soi.assign_coords(
            time=pd.to_datetime(soi["time"].values)
        )
    except Exception:
        pass

    soi = soi.sel(
        time=slice(
            f"{P1_START}-01-01",
            f"{P2_END}-12-31",
        )
    )

    soi_gs = (
        soi
        .where(
            soi["time.month"].isin(GS_MONTHS),
            drop=True,
        )
        .groupby("time.year")
        .mean("time")
    )


    rows = []
    extreme_years = {}

    for st in STATE_ORDER:
        col = f"{st}_detrended"
        tmp = class_ts[["year", col]].dropna().copy()

        if st in ["WW", "DW"]:
            threshold = tmp[col].quantile(EXTREME_QUANTILE)
            yrs = tmp.loc[
                tmp[col] >= threshold,
                "year",
            ].astype(int).tolist()
            direction = "upper"
        else:
            threshold = tmp[col].quantile(1.0 - EXTREME_QUANTILE)
            yrs = tmp.loc[
                tmp[col] <= threshold,
                "year",
            ].astype(int).tolist()
            direction = "lower"

        extreme_years[st] = yrs

        rows.append({
            "state": st,
            "selection": direction + "_quartile_detrended_storage",
            "threshold": threshold,
            "years": ",".join(map(str, yrs)),
            "n_years": len(yrs),
        })

    soi_years_df = pd.DataFrame(rows)

    safe_to_csv(
        soi_years_df,
        OUT_DIR / "soi_state_consistent_extreme_years.csv",
    )


    soi_overlap_df = pairwise_year_overlap_table(extreme_years)

    safe_to_csv(
        soi_overlap_df,
        OUT_DIR / "soi_selected_year_overlap_diagnostics.csv",
    )

    for _, rr in soi_overlap_df.iterrows():
        if (
            rr["state_1"] != rr["state_2"]
            and np.isfinite(rr["jaccard"])
            and rr["jaccard"] >= SOI_JACCARD_WARN
        ):
            print(
                f"WARNING: SOI year-set overlap is high for "
                f"{rr['state_1']} vs {rr['state_2']}: "
                f"Jaccard={rr['jaccard']:.2f}"
            )

    available_years = set(
        int(y) for y in soi_gs["year"].values
    )

    soi_comps = {}

    for st, yrs in extreme_years.items():
        yrs_common = [
            y for y in yrs
            if y in available_years
        ]

        if yrs_common:
            comp = soi_gs.sel(
                year=yrs_common
            ).mean("year")

            soi_comps[st] = comp

            comp.to_dataframe(
                name=f"SOI_composite_{st}"
            ).reset_index().to_csv(
                OUT_DIR / f"soi_composite_{st}_grid.csv",
                index=False,
            )

        else:
            soi_comps[st] = None

    soi_spatial_corr_df = soi_composite_spatial_correlation(
        soi_comps
    )

    safe_to_csv(
        soi_spatial_corr_df,
        OUT_DIR / "soi_composite_spatial_correlation_diagnostics.csv",
    )

    vals = []
    for comp in soi_comps.values():
        if comp is not None:
            arr = np.asarray(comp.values)
            vals.append(arr[np.isfinite(arr)])

    if vals:
        vals = np.concatenate(vals)
        lim = np.nanpercentile(np.abs(vals), 98)

        if np.isfinite(lim) and lim > 0:
            soi_vmin, soi_vmax = -lim, lim


def significance_stars(p):
    if not np.isfinite(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def plot_base_map(ax):
    setup_map_ax(ax)

    cmap = ListedColormap([
        CLASS_COLORS_BASE[1],
        CLASS_COLORS_BASE[2],
    ])

    norm = mcolors.BoundaryNorm(
        [0.5, 1.5, 2.5],
        cmap.N,
    )

    im = ax.pcolormesh(
        class_base["lon"],
        class_base["lat"],
        class_base,
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
        shading="auto",
        rasterized=MAP_RASTERIZED_IN_VECTOR,
    )

    plot_mask(
        ax,
        class_base["lon"].values,
        class_base["lat"].values,
        sig_mask_p1_fdr,
        step=3,
    )

    lookup = dict(
        zip(
            pcts_base_df["class_label"],
            pcts_base_df["area_weighted_percent"],
        )
    )

    ax.set_title(
        "Direct-storage trend direction (1979–2000)",
        fontsize=10,
        fontweight="bold",
    )

    cb = plt.colorbar(
        im,
        ax=ax,
        orientation="horizontal",
        fraction=0.045,
        pad=0.045,
    )

    cb.set_ticks([1, 2])
    cb.set_ticklabels([
        f"Wetting {lookup.get('Wetting', np.nan):.1f}%",
        f"Drying {lookup.get('Drying', np.nan):.1f}%",
    ])

    cb.ax.tick_params(labelsize=7)


def plot_transition_map(ax):
    setup_map_ax(ax)

    cmap = ListedColormap([
        CLASS_COLORS_TRANS[1],
        CLASS_COLORS_TRANS[2],
        CLASS_COLORS_TRANS[3],
        CLASS_COLORS_TRANS[4],
    ])

    norm = mcolors.BoundaryNorm(
        [0.5, 1.5, 2.5, 3.5, 4.5],
        cmap.N,
    )

    im = ax.pcolormesh(
        class_trans["lon"],
        class_trans["lat"],
        class_trans,
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
        shading="auto",
        rasterized=MAP_RASTERIZED_IN_VECTOR,
    )

    plot_mask(
        ax,
        class_trans["lon"].values,
        class_trans["lat"].values,
        sig_mask_transition_fdr,
        step=3,
    )

    lookup = dict(
        zip(
            pcts_trans_df["class_label"],
            pcts_trans_df["area_weighted_percent"],
        )
    )

    ax.set_title(
        "Direct-storage DDWW transition (1979–2000 to 2001–2023)",
        fontsize=10,
        fontweight="bold",
    )

    cb = plt.colorbar(
        im,
        ax=ax,
        orientation="horizontal",
        fraction=0.045,
        pad=0.045,
    )

    cb.set_ticks([1, 2, 3, 4])
    cb.set_ticklabels([
        f"WW {lookup.get('WW', np.nan):.1f}%",
        f"DD {lookup.get('DD', np.nan):.1f}%",
        f"WD {lookup.get('WD', np.nan):.1f}%",
        f"DW {lookup.get('DW', np.nan):.1f}%",
    ])

    cb.ax.tick_params(labelsize=7)


def plot_class_storage_timeseries(ax):
    for st in STATE_ORDER:
        ax.plot(
            class_ts["year"],
            class_ts[f"{st}_z"],
            linewidth=1.7,
            color=STATE_COLORS[st],
            label=st,
        )

    ax.axhline(
        0,
        color="black",
        linewidth=0.7,
        alpha=0.6,
    )

    ax.axvline(
        2001,
        color="black",
        linewidth=0.8,
        linestyle="--",
        alpha=0.65,
    )

    ax.set_ylabel(
        "Class-mean standardized direct storage (z)",
        fontsize=9,
    )
    ax.set_xlabel("Year", fontsize=9)

    ax.set_title(
        "Annual class-mean direct-storage anomaly within fixed DDWW regions",
        fontsize=10,
        fontweight="bold",
    )

    ax.grid(alpha=0.25, linewidth=0.5)
    ax.tick_params(labelsize=8)
    ax.legend(
        ncol=4,
        fontsize=8,
        frameon=True,
        loc="upper center",
    )


def plot_corr_heatmap(
    ax,
    corr_df,
    value_col="pearson_r",
    p_col="pearson_q_ar1_fdr",
    title="Correlation",
):
    indices = ["pdo", "ao", "noi"]

    mat = (
        corr_df
        .pivot(
            index="state",
            columns="index",
            values=value_col,
        )
        .reindex(STATE_ORDER)[indices]
    )

    pmat = (
        corr_df
        .pivot(
            index="state",
            columns="index",
            values=p_col,
        )
        .reindex(STATE_ORDER)[indices]
    )

    im = ax.imshow(
        mat.values,
        cmap="RdBu_r",
        vmin=-1,
        vmax=1,
        aspect="auto",
    )

    ax.set_xticks(np.arange(len(indices)))
    ax.set_yticks(np.arange(len(STATE_ORDER)))

    ax.set_xticklabels(
        [x.upper() for x in indices],
        fontsize=8,
    )

    ax.set_yticklabels(
        STATE_ORDER,
        fontsize=8,
    )

    ax.set_title(
        title,
        fontsize=10,
        fontweight="bold",
    )

    for i in range(len(STATE_ORDER)):
        for j in range(len(indices)):
            val = mat.values[i, j]
            p = pmat.values[i, j]

            if np.isfinite(val):
                ax.text(
                    j,
                    i,
                    f"{val:.2f}{significance_stars(p)}",
                    ha="center",
                    va="center",
                    fontsize=7,
                )

    ax.tick_params(length=0)

    cb = plt.colorbar(
        im,
        ax=ax,
        orientation="vertical",
        fraction=0.05,
        pad=0.03,
    )

    cb.set_label("r", fontsize=8)
    cb.ax.tick_params(labelsize=7)


def plot_phase_heatmap(ax, phase_df):
    indices = ["pdo", "ao", "noi"]

    mat = (
        phase_df
        .pivot(
            index="state",
            columns="index",
            values="pos_minus_neg",
        )
        .reindex(STATE_ORDER)[indices]
    )

    vmax = np.nanmax(np.abs(mat.values))

    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    im = ax.imshow(
        mat.values,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        aspect="auto",
    )

    ax.set_xticks(np.arange(len(indices)))
    ax.set_yticks(np.arange(len(STATE_ORDER)))

    ax.set_xticklabels(
        [x.upper() for x in indices],
        fontsize=8,
    )

    ax.set_yticklabels(
        STATE_ORDER,
        fontsize=8,
    )

    ax.set_title(
        "Positive − negative standardized climate phase",
        fontsize=10,
        fontweight="bold",
    )

    for i in range(len(STATE_ORDER)):
        for j in range(len(indices)):
            val = mat.values[i, j]

            if np.isfinite(val):
                ax.text(
                    j,
                    i,
                    f"{val:+.2f}",
                    ha="center",
                    va="center",
                    fontsize=7,
                )

    ax.tick_params(length=0)

    cb = plt.colorbar(
        im,
        ax=ax,
        orientation="vertical",
        fraction=0.05,
        pad=0.03,
    )

    cb.set_label(
        "Detrended storage difference (z)",
        fontsize=8,
    )

    cb.ax.tick_params(labelsize=7)


def plot_soi_composite(ax, comp, st):
    setup_map_ax(ax)

    if comp is not None:
        im = ax.pcolormesh(
            comp["lon"],
            comp["lat"],
            comp,
            transform=ccrs.PlateCarree(),
            cmap="RdBu_r",
            vmin=soi_vmin,
            vmax=soi_vmax,
            shading="auto",
            rasterized=MAP_RASTERIZED_IN_VECTOR,
        )
    else:
        im = None
        ax.text(
            0.5,
            0.5,
            "No SOI composite",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=9,
        )

    if st in ["WW", "DW"]:
        descriptor = "wet-anomaly years"
    else:
        descriptor = "dry-anomaly years"

    ax.set_title(
        f"SOI: {st} {descriptor}",
        fontsize=9,
        fontweight="bold",
    )

    return im


def plot_area_percent_bars(ax):
    labels = ["Wetting", "Drying", "WW", "DD", "WD", "DW"]

    lookup_base = dict(
        zip(
            pcts_base_df["class_label"],
            pcts_base_df["area_weighted_percent"],
        )
    )

    lookup_trans = dict(
        zip(
            pcts_trans_df["class_label"],
            pcts_trans_df["area_weighted_percent"],
        )
    )

    values = [
        lookup_base.get("Wetting", np.nan),
        lookup_base.get("Drying", np.nan),
        lookup_trans.get("WW", np.nan),
        lookup_trans.get("DD", np.nan),
        lookup_trans.get("WD", np.nan),
        lookup_trans.get("DW", np.nan),
    ]

    colors = [
        CLASS_COLORS_BASE[1],
        CLASS_COLORS_BASE[2],
        CLASS_COLORS_TRANS[1],
        CLASS_COLORS_TRANS[2],
        CLASS_COLORS_TRANS[3],
        CLASS_COLORS_TRANS[4],
    ]

    ax.bar(
        labels,
        values,
        color=colors,
        edgecolor="black",
        linewidth=0.4,
    )

    ax.set_ylabel(
        "Area-weighted fraction (%)",
        fontsize=9,
    )

    ax.set_title(
        "Mapped sign-class percentages",
        fontsize=10,
        fontweight="bold",
    )

    ax.tick_params(
        axis="x",
        rotation=35,
        labelsize=8,
    )

    ax.tick_params(
        axis="y",
        labelsize=8,
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )



print("Creating figure...")

fig = plt.figure(figsize=PUB_FIGSIZE)

gs = gridspec.GridSpec(
    nrows=3,
    ncols=4,
    figure=fig,
    height_ratios=[1.15, 0.95, 0.90],
    width_ratios=[1, 1, 1, 1],
    hspace=0.35,
    wspace=0.28,
)

# a. P1 direction
axA = fig.add_subplot(
    gs[0, 0:2],
    projection=ccrs.PlateCarree(),
)
plot_base_map(axA)
add_panel_label(axA, "a")

# b. Long-term DDWW transition
axB = fig.add_subplot(
    gs[0, 2:4],
    projection=ccrs.PlateCarree(),
)
plot_transition_map(axB)
add_panel_label(axB, "b")

# c. Annual class-conditioned direct-storage anomaly
axC = fig.add_subplot(gs[1, 0:2])
plot_class_storage_timeseries(axC)
add_panel_label(axC, "c", x=0.005, y=0.98)

# d. Annual correlations
axD = fig.add_subplot(gs[1, 2])
plot_corr_heatmap(
    axD,
    corr_annual,
    value_col="pearson_r",
    p_col="pearson_q_ar1_fdr",
    title="Annual climate correlation (AR1 + FDR)",
)
add_panel_label(axD, "d", x=-0.17, y=1.10)

# e. Phase contrasts
axE = fig.add_subplot(gs[1, 3])
plot_phase_heatmap(axE, phase_annual)
add_panel_label(axE, "e", x=-0.17, y=1.10)

# Bottom row
if soi_comps is not None:
    soi_axes = []
    soi_ims = []

    for j, st in enumerate(STATE_ORDER):
        ax = fig.add_subplot(
            gs[2, j],
            projection=ccrs.PlateCarree(),
        )

        im = plot_soi_composite(
            ax,
            soi_comps.get(st),
            st,
        )

        add_panel_label(
            ax,
            chr(ord("f") + j),
        )

        soi_axes.append(ax)

        if im is not None:
            soi_ims.append(im)

    if soi_ims:
        cbar = fig.colorbar(
            soi_ims[0],
            ax=soi_axes,
            orientation="horizontal",
            fraction=0.045,
            pad=0.07,
        )

        cbar.set_label(
            "SOI anomaly",
            fontsize=8,
        )

        cbar.ax.tick_params(labelsize=7)

else:
    axF = fig.add_subplot(gs[2, 0])
    plot_area_percent_bars(axF)
    add_panel_label(axF, "f", x=-0.12, y=1.08)

    axG = fig.add_subplot(gs[2, 1:3])

    for col, lab in [
        ("pdo", "PDO"),
        ("ao", "AO"),
        ("noi", "NOI"),
    ]:
        merged = pd.DataFrame({
            "year": np.arange(P1_START, P2_END + 1)
        })

        src = {
            "pdo": pdo_ann,
            "ao": ao_ann,
            "noi": noi_ann,
        }[col]

        merged = merged.merge(
            src[["year", col]],
            on="year",
            how="left",
        )

        vals = merged[col].astype(float)
        sd = vals.std(skipna=True)

        if np.isfinite(sd) and sd > 0:
            z = (vals - vals.mean(skipna=True)) / sd
            axG.plot(
                merged["year"],
                z,
                linewidth=1.4,
                label=lab,
            )

    axG.axhline(
        0,
        color="black",
        linewidth=0.7,
    )

    axG.axvline(
        2001,
        color="black",
        linestyle="--",
        linewidth=0.8,
        alpha=0.65,
    )

    axG.set_title(
        "Standardized climate indices",
        fontsize=10,
        fontweight="bold",
    )

    axG.set_xlabel("Year", fontsize=9)
    axG.set_ylabel("Standardized index", fontsize=9)
    axG.grid(alpha=0.25)
    axG.legend(ncol=3, fontsize=8)
    add_panel_label(axG, "g", x=-0.04, y=1.08)

    axH = fig.add_subplot(gs[2, 3])
    plot_corr_heatmap(
        axH,
        corr_gs,
        value_col="pearson_r",
        p_col="pearson_q_ar1_fdr",
        title="Growing-season correlation",
    )
    add_panel_label(axH, "h", x=-0.17, y=1.10)

fig.suptitle(
    "Large-scale climate associations with revised direct-storage DDWW regions",
    fontsize=14,
    fontweight="bold",
    y=0.985,
)

fig.text(
    0.5,
    0.012,
    "DDWW classes are assigned from direct-storage trend signs",
    ha="center",
    va="bottom",
    fontsize=8,
)

save_figure(
    fig,
    OUT_DIR / "figure_R2_FINAL_STATS",
    dpi=FIG_DPI,
)

plt.close(fig)


# =============================================================================
# 11. INDIVIDUAL FIGURES
# =============================================================================

# Transition maps
fig = plt.figure(figsize=(14, 8))
gs2 = gridspec.GridSpec(
    2, 1,
    figure=fig,
    hspace=0.22,
)

ax1 = fig.add_subplot(
    gs2[0, 0],
    projection=ccrs.PlateCarree(),
)
plot_base_map(ax1)
add_panel_label(ax1, "a")

ax2 = fig.add_subplot(
    gs2[1, 0],
    projection=ccrs.PlateCarree(),
)
plot_transition_map(ax2)
add_panel_label(ax2, "b")

save_figure(
    fig,
    OUT_DIR / "figure_DIRECT_STORAGE_transition_maps",
    dpi=FIG_DPI,
)

plt.close(fig)

# Class-conditioned annual storage
fig, ax = plt.subplots(figsize=(10, 4.2))
plot_class_storage_timeseries(ax)
plt.tight_layout()

save_figure(
    fig,
    OUT_DIR / "figure_annual_direct_storage_by_DDWW_class",
    dpi=FIG_DPI,
)

plt.close(fig)

# Correlation heatmaps
fig, axes = plt.subplots(
    1, 2,
    figsize=(9, 4),
)

plot_corr_heatmap(
    axes[0],
    corr_annual,
    value_col="pearson_r",
    p_col="pearson_q_ar1_fdr",
    title="Annual correlation (AR1 + FDR)",
)

plot_corr_heatmap(
    axes[1],
    corr_gs,
    value_col="pearson_r",
    p_col="pearson_q_ar1_fdr",
    title="Growing-season correlation (AR1 + FDR)",
)

plt.tight_layout()

save_figure(
    fig,
    OUT_DIR / "figure_direct_storage_climate_correlation_heatmaps",
    dpi=FIG_DPI,
)

plt.close(fig)

# Phase contrast
fig, ax = plt.subplots(figsize=(4.8, 4.2))
plot_phase_heatmap(ax, phase_annual)
plt.tight_layout()

save_figure(
    fig,
    OUT_DIR / "figure_direct_storage_climate_phase_contrasts",
    dpi=FIG_DPI,
)

plt.close(fig)


# =============================================================================
# 12. METHOD METADATA
# =============================================================================

method_rows = [
    ("primary_state", "TerraClimate soil + SWE"),
    ("historical_period_1", "1979-2000"),
    ("historical_period_2", "2001-2023"),
    (
        "DDWW_classification",
        
    ),
    (
        "significance",
        
    ),
    (
        "annual_climate_response",
        "raw annual soil+SWE storage is area-averaged within each fixed DDWW region first, then the class-mean series is standardized to 1979-2000",
    ),
    (
        "climate_correlation",

    ),
    (
        "phase_composite",
        f"PDO/AO/NOI standardized over study years; positive phase z >= +{PHASE_THRESHOLD}, negative phase z <= -{PHASE_THRESHOLD}; phase contrasts treated descriptively",
    ),
    (
        "SOI_composite",
        
    ),
    ("area_weighting", "actual ice-free land area (m2)"),
]

pd.DataFrame(
    method_rows,
    columns=["item", "value"],
).to_csv(
    OUT_DIR / "analysis_method_metadata_DIRECT_STORAGE_R2_FINAL_STATS.csv",
    index=False,
)

print("\nDone.")
print("Outputs saved in:")
print(OUT_DIR)
print("\nMain figure:")
print(
    OUT_DIR
    / "DIRECT_STORAGE_climate_associations_R2_FINAL_STATS.svg"
)
